In [ ]:
# Example 1: URL parsing and manipulation (no network)
import httpx2

url = httpx2.URL("https://example.com/path?foo=1&bar=2")
print(f"scheme: {url.scheme}")
print(f"host:   {url.host}")
print(f"path:   {url.path}")
print(f"params: {dict(url.params)}")

updated = url.copy_with(path="/new")
print(f"updated path: {updated.path}")

assert url.scheme == "https"
assert url.host == "example.com"
assert updated.path == "/new"

In [ ]:
# Example 2: Headers (case-insensitive multi-dict)
import httpx2

headers = httpx2.Headers({"Content-Type": "application/json", "X-Custom": "v1"})
print(f"content-type: {headers['content-type']}")
print(f"CONTENT-TYPE: {headers['CONTENT-TYPE']}")

headers["Authorization"] = "Bearer token"
print(f"all keys: {list(headers.keys())}")

assert headers["content-type"] == "application/json"
assert headers["authorization"] == "Bearer token"
assert "X-Custom" in headers

In [ ]:
# Example 3: Request construction (no network send)
import httpx2

req = httpx2.Request(
    "POST",
    "https://api.example.com/v1/items",
    json={"name": "widget", "qty": 3},
    headers={"X-Trace-Id": "abc123"},
)
print(f"method: {req.method}")
print(f"url:    {req.url}")
print(f"content-type: {req.headers.get('content-type')}")
print(f"body bytes: {req.content[:60]}...")

assert req.method == "POST"
assert req.headers["x-trace-id"] == "abc123"
assert b"widget" in req.content

In [ ]:
# Example 4: MockTransport — exercise Client without real network
import httpx2

def handler(request: httpx2.Request) -> httpx2.Response:
    return httpx2.Response(
        200,
        json={"echo_path": request.url.path, "method": request.method},
    )

transport = httpx2.MockTransport(handler)
with httpx2.Client(transport=transport, base_url="https://fake.test") as client:
    resp = client.get("/hello")
    print(f"status: {resp.status_code}")
    print(f"json:   {resp.json()}")

assert resp.status_code == 200
assert resp.json() == {"echo_path": "/hello", "method": "GET"}

In [ ]:
# Example 5: AsyncClient + MockTransport (top-level await; no asyncio.run)
import httpx2

async def async_handler(request: httpx2.Request) -> httpx2.Response:
    return httpx2.Response(201, json={"created": True, "url": str(request.url)})

transport = httpx2.MockTransport(async_handler)
async with httpx2.AsyncClient(transport=transport, base_url="https://fake.test") as client:
    resp = await client.post("/items", json={"name": "x"})
    body = resp.json()

print(f"status: {resp.status_code}")
print(f"body:   {body}")

assert resp.status_code == 201
assert body["created"] is True